# 付録：ハンズオン③ 深掘り編

**前提**: `ch1_03_llm_inference.ipynb`（本編）を終えていること
**推奨所要時間**: 約 45 分（興味のある付録だけつまみ食いして構いません）

---

本編は 30 分で「Attention を作って LLM を動かす」ところまでを一気に通しました。
この付録には、本編から外した **もう一段深い内容** を収めてあります。

| 付録 | 内容 | 本編との関係 |
|---|---|---|
| **A** | $\sqrt{d_k}$ で割る理由／学習前のランダムな重み／日本語のトークン効率 | Part 2 の補足 |
| **B** | Multi-Head Attention を作る | Part 3 の続き |
| **C** | Transformer ブロックを `nn.Module` で組み立てる | Part 4 の中身 |
| **D** | 層ごとの Attention の違いと Attention Sink | Part 5 の続き |
| **E** | 全位置ぶんの予測／KV キャッシュ／サンプリングの自作実装 | Part 6 の中身 |
| **F** | 推論パラメータの定量評価（TTR・Jaccard・repetition_penalty） | Part 7 の続き |

**各付録は独立しています。** 次の「準備」セルさえ実行すれば、
どこから始めても動きます。

---

## 準備：本編で定義したものをまとめて再現する

本編で書いたコードのうち、この付録で使うものだけを 1 セルにまとめてあります。
**まずこのセルを実行してください。**（中身は本編とまったく同じです）

In [ ]:
# 実行時間: 初回は数分（モデルのダウンロード）、2回目以降は約30秒
import os

if os.path.exists('/data/shared/models'):
    os.environ['HF_HOME'] = '/data/shared/models'

import math
import time
import unicodedata
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from itertools import combinations
from transformers import AutoTokenizer, AutoModelForCausalLM

from matplotlib import rcParams
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Noto Sans CJK JP', 'IPAexGothic', 'IPAPGothic', 'VL PGothic',
                               'Hiragino Maru Gothic Pro', 'Yu Gothic', 'Meirio', 'Takao']
rcParams['axes.unicode_minus'] = False

GREEN, AMBER, RED = '#0D4A38', '#7C5C00', '#991B1B'
colors = [GREEN, AMBER, RED]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
_ = torch.manual_seed(42)


def pad(s, width: int, right: bool = False) -> str:
    """全角文字を2文字ぶんとして数え、表示幅をそろえる"""
    s = str(s)
    w = sum(2 if unicodedata.east_asian_width(c) in 'WF' else 1 for c in s)
    space = ' ' * max(width - w, 0)
    return space + s if right else s + space


# ── モデル ─────────────────────────────────────────────
MODEL_NAME = 'meta-llama/Llama-3.2-1B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, clean_up_tokenization_spaces=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float16, device_map='auto', attn_implementation='eager')
model.eval()
cfg = model.config
E = model.get_input_embeddings().weight.detach()

# ── Part 2 のミニチュア世界 ────────────────────────────
toy_tokens    = ['ネコ', 'が', 'サカナ', '食べた']
feature_names = ['名詞性', '動詞性', '生き物性', '食べ物性']
X = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],   # ネコ
    [0.2, 0.0, 0.0, 0.0],   # が
    [1.0, 0.0, 0.5, 1.0],   # サカナ
    [0.0, 1.0, 0.0, 0.0],   # 食べた
])
W_Q = torch.tensor([[0., 0.], [0., 3.], [3., 0.], [0., 0.]])
W_K = torch.tensor([[0., 0.], [0., 0.], [3., 0.], [0., 3.]])
W_V = torch.eye(4)
Q, K, V = X @ W_Q, X @ W_K, X @ W_V


def scaled_dot_product_attention(Q, K, V):
    """Self-Attention の本体。attention の出力と注目度の重みを返す

    Q: (トークン数, d_k)   各トークンの問い合わせ
    K: (トークン数, d_k)   各トークンの見出し
    V: (トークン数, d_v)   各トークンの中身
    """
    d_k = Q.shape[-1]

    # ① 全ペアの相性スコア: (トークン数, トークン数)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

    # ② 行ごとに softmax して「合計1の注目度」に変換
    weights = torch.softmax(scores, dim=-1)

    # ③ 注目度を重みにして V の加重平均を取る
    output = weights @ V

    return output, weights


def causal_self_attention(Q, K, V):
    """未来のトークンを見られないようにした Self-Attention"""
    d_k = Q.shape[-1]
    T = Q.shape[-2]

    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

    # 上三角（未来の位置）を -inf にする
    mask = torch.tril(torch.ones(T, T, device=Q.device, dtype=torch.bool))
    scores = scores.masked_fill(~mask, float('-inf'))

    weights = torch.softmax(scores, dim=-1)
    return weights @ V, weights


def plot_attention(weights, x_labels, y_labels, title, ax=None, cmap='Greens', show_values=True):
    """注目度行列をヒートマップで描く共通関数（このあと何度も使う）"""
    created = ax is None
    if created:
        fig, ax = plt.subplots(figsize=(5.5, 4.5))
    w = weights.detach().cpu().float()
    im = ax.imshow(w, cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(range(len(x_labels)), x_labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(y_labels)), y_labels, fontsize=9)
    if show_values:
        for i in range(w.shape[0]):
            for j in range(w.shape[1]):
                v = float(w[i, j])
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        fontsize=8, color='white' if v > 0.5 else '#333333')
    ax.set_title(title, fontsize=11, pad=10)
    ax.set_xlabel('見られる側（情報の出し手）', fontsize=9)
    ax.set_ylabel('見る側（情報の受け手）', fontsize=9)
    if created:
        plt.tight_layout()
        plt.show()


attn_out, attn_w = scaled_dot_product_attention(Q, K, V)

# ── Part 5 で使った実モデルの注目度 ────────────────────
sentence = 'The cat did not cross the street because it was too tired'
_enc = tokenizer(sentence, return_tensors='pt').to(model.device)
tok_labels = [tokenizer.decode([i]) for i in _enc['input_ids'][0]]
with torch.no_grad():
    A = torch.stack(model(**_enc, output_attentions=True).attentions).float().cpu()[:, 0]


def next_token_distribution(text: str):
    """文字列を入力し、次のトークンの確率分布を返す"""
    enc = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        logits = model(**enc).logits          # (1, トークン数, 語彙数)
    last_logits = logits[0, -1].float()       # 最後の位置＝「次に来る語」の予測
    return torch.softmax(last_logits, dim=-1).cpu(), last_logits.cpu()


def greedy_generate(text: str, max_new_tokens: int = 25, verbose: bool = True) -> str:
    """model.generate() を使わず、自分で 1 トークンずつ生成する"""
    ids = tokenizer(text, return_tensors='pt')['input_ids'].to(model.device)

    for step in range(max_new_tokens):
        with torch.no_grad():
            logits = model(ids).logits[0, -1]     # 次のトークンの予測

        next_id = int(logits.argmax())            # 最も確率の高いものを選ぶ（＝貪欲法）

        if next_id == tokenizer.eos_token_id:
            if verbose:
                print('\n[終端トークンが出たので停止]')
            break

        # 選んだトークンを末尾に足して、もう一周
        ids = torch.cat([ids, torch.tensor([[next_id]], device=ids.device)], dim=1)

        if verbose:
            print(tokenizer.decode([next_id]), end='', flush=True)

    return tokenizer.decode(ids[0], skip_special_tokens=True)


def build_prompt(user_message: str, system: str = 'You are a helpful assistant.') -> str:
    """Llama-3 の chat template に合わせたプロンプトを組み立てる"""
    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': user_message},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate_text(
    prompt: str,
    max_new_tokens: int = 150,
    do_sample: bool = True,
    temperature: float = 0.7,
    top_p: float = 0.9,
    repetition_penalty: float = 1.1,
) -> str:
    """プロンプトを受け取り、生成テキスト（プロンプト部分を除く）を返す"""
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.eos_token_id,
        )

    # プロンプト部分を除いた生成トークンだけデコード
    generated_ids = outputs[0][input_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


print('準備完了')
print(f'  デバイス     : {device}')
print(f'  注目度テンソル: {tuple(A.shape)}  (層, ヘッド, 見る側, 見られる側)')

---

# 付録 A: Attention の細部

## A-1. なぜ $\sqrt{d_k}$ で割るのか

本編では一行の注釈で済ませた $\sqrt{d_k}$ を、実験で確かめます。

**次元が増えると、ベクトルは長くなります。**
適当に取ってきた 10 次元のベクトルと 100 次元のベクトルなら、後者のほうが長い。
足し合わせる成分の数が増えるからです。

内積はその長さに引きずられるので、**次元数が大きいだけでスコアが大きくなってしまいます**。
「次元が高いから類似度が高い」というのは、明らかにおかしい。

In [ ]:
# 実行時間: 数秒

print('ランダムなベクトル同士の内積の標準偏差\n')
print(f'{"次元 d_k":>10}{"割らない場合":>16}{"√d_k で割った場合":>20}')
print('-' * 48)
for d in [8, 64, 512, 2048]:
    q = torch.randn(3000, d)
    k = torch.randn(3000, d)
    dots = (q * k).sum(dim=-1)
    print(f'{d:>10}{float(dots.std()):>16.2f}{float((dots / math.sqrt(d)).std()):>20.2f}')

# softmax の尖り方を比較
d_demo = 2048
q_demo = torch.randn(1, d_demo)
k_demo = torch.randn(6, d_demo)
raw = (q_demo @ k_demo.T)[0]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, (vals, ttl, col) in zip(axes, [
    (torch.softmax(raw, dim=-1), f'√d_k で割らない場合（d_k={d_demo}）', RED),
    (torch.softmax(raw / math.sqrt(d_demo), dim=-1), f'√d_k で割った場合（d_k={d_demo}）', GREEN),
]):
    ax.bar(range(6), vals, color=col, width=0.6)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('候補トークン')
    ax.set_ylabel('注目度')
    ax.set_title(ttl, fontsize=10)
plt.tight_layout()
plt.show()

print('\n割らないと 1 箇所に全振りされ、他のトークンの情報が一切届かなくなる。')

## A-2. 重みが「学習されている」とはどういうことか

本編 Part 2-3 では $W_Q, W_K$ を手で設計しました。
では、**学習前のランダムな重み**だとどうなるでしょうか。

In [ ]:
# 実行時間: 数秒

torch.manual_seed(0)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

# 1枚目: 手で設計した重み
plot_attention(attn_w, toy_tokens, toy_tokens, '手で設計した重み', ax=axes[0])

# 2〜3枚目: ランダムな重み（＝学習前の状態）
for k in range(2):
    W_Q_rand = torch.randn(4, 2)
    W_K_rand = torch.randn(4, 2)
    _, w_rand = scaled_dot_product_attention(X @ W_Q_rand, X @ W_K_rand, V)
    plot_attention(w_rand, toy_tokens, toy_tokens, f'ランダムな重み #{k + 1}', ax=axes[1 + k])

plt.tight_layout()
plt.show()

print('ランダムな重みでは、注目先に意味がない（毎回バラバラ）。')
print('大量のテキストで「次の単語を当てる」練習を繰り返すうちに、')
print('W_Q・W_K が「動詞は目的語を見る」といった規則を自力で獲得していく。')
print('→ Part 5 で、本物の Llama が獲得した重みの結果を実際に見る。')

## A-3. 日本語は英語より多くのトークンを消費する

トークン数は課金額・処理時間・コンテキスト長の上限すべてに直結します。
同じ意味でも日本語のほうがトークンを食う、というのは実務で効いてくる感覚です。

In [ ]:
# 実行時間: 数秒
# 同じ意味の文でトークン数を比較する

pairs = [
    ('Artificial intelligence is changing the world.',
     '人工知能は世界を変えつつある。'),
    ('Please summarize the following document in three sentences.',
     '以下の文書を3文で要約してください。'),
]

print(pad('英語', 8, right=True) + ' / ' + pad('日本語', 8, right=True) + '   文')
print('-' * 70)
for en, ja in pairs:
    n_en = len(tokenizer.encode(en, add_special_tokens=False))
    n_ja = len(tokenizer.encode(ja, add_special_tokens=False))
    print(f'{n_en:>8} / {n_ja:>8}   {en}')
    print(' ' * 21 + f'{ja}   （{n_ja / n_en:.1f} 倍）')

---

# 付録 B: Multi-Head Attention を作る

## B-1. Multi-Head：複数の視点で同時に見る

Part 2 で作った Attention は「動詞は食べ物を探す」という **1 種類の関係**しか捉えられません。
しかし言語には、同時に見るべき関係がいくつもあります。

- 「それ」が指すのは誰か（共参照）
- 動詞の目的語はどれか（係り受け）
- 主語はどれか

2-3 で見たとおり、$W_Q$ と $W_K$ は「$X$ をどうねじってからぶつけるか」を決めていました。
**ねじり方を変えれば、測れる関係も変わります。**

そこで、**$W_Q, W_K, W_V$ の組を複数用意して並列に走らせます**。この 1 組を **ヘッド（head）** と呼びます。

> **Multi-Head Attention とは、同じ文の類似度を「いろんな角度から測る」ことです。**

各ヘッドは別々の関係を学習し、最後に結果を結合します。

ここでは「食べ物を探すヘッド」と「生き物を探すヘッド」を手で作り、違いを見てみます。

In [ ]:
# 実行時間: 数秒

# ヘッド A: 動詞が「食べ物」を探す（Part 2 と同じ設計）
W_Q_A = torch.tensor([[0., 0.], [0., 3.], [3., 0.], [0., 0.]])
W_K_A = torch.tensor([[0., 0.], [0., 0.], [3., 0.], [0., 3.]])

# ヘッド B: すべてのトークンが「生き物」を探す（主語を見つけるヘッドのイメージ）
W_Q_B = torch.tensor([[3., 0.], [3., 0.], [0., 0.], [0., 0.]])
W_K_B = torch.tensor([[0., 0.], [0., 0.], [3., 0.], [0., 0.]])

out_A, w_A = causal_self_attention(X @ W_Q_A, X @ W_K_A, V)
out_B, w_B = causal_self_attention(X @ W_Q_B, X @ W_K_B, V)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
plot_attention(w_A, toy_tokens, toy_tokens, 'ヘッドA：目的語（食べ物）を探す', ax=axes[0])
plot_attention(w_B, toy_tokens, toy_tokens, 'ヘッドB：主語（生き物）を探す', ax=axes[1])
plt.tight_layout()
plt.show()

# 各ヘッドの出力を結合するのが Multi-Head Attention
multi_head_out = torch.cat([out_A, out_B], dim=-1)
print(f'ヘッドA の出力形状 : {tuple(out_A.shape)}')
print(f'ヘッドB の出力形状 : {tuple(out_B.shape)}')
print(f'結合後の出力形状   : {tuple(multi_head_out.shape)}  ← これを線形層で元の次元に戻す')

同じ文でも、ヘッドによって**まったく違う場所を見ている**のがポイントです。

実装上は、ヘッドごとに別の行列を持つのではなく、
**$d_{model}$ 次元を ヘッド数で割って分割する**方式を取ります（計算量を増やさないため）。
Llama-3.2-1B の場合はこうなります。

In [ ]:
# 実行時間: 数秒

d_model  = cfg.hidden_size
n_heads  = cfg.num_attention_heads
n_kv     = cfg.num_key_value_heads
d_head   = d_model // n_heads

print(f'隠れ次元 d_model      : {d_model}')
print(f'ヘッド数 n_heads      : {n_heads}')
print(f'1ヘッドの次元 d_head  : {d_model} / {n_heads} = {d_head}')
print(f'KV ヘッド数           : {n_kv}')
print()
print(f'→ 各層で {n_heads} 種類の「関係の見方」が並列に走っている。')
print(f'→ {cfg.num_hidden_layers} 層あるので、モデル全体では '
      f'{cfg.num_hidden_layers * n_heads} 個のヘッドが動いている。')
print()
print(f'※ KV ヘッドが {n_kv} 個しかないのは GQA（Grouped Query Attention）という省メモリ手法。')
print(f'   {n_heads // n_kv} 個の Query ヘッドで 1 組の K・V を共有し、推論時のメモリを削減している。')

---

# 付録 C: Transformer ブロックを組み立てる

本編 Part 4 では、本物の Llama の 1 層を `print()` して部品を確認しただけでした。
ここでは、その部品を **自分で `nn.Module` として組み立てます。**

| 部品 | 役割 |
|---|---|
| **正規化（LayerNorm / RMSNorm）** | 入力のスケールを揃え、学習を安定させる |
| **Multi-Head Attention** | トークン間で情報をやりとりする（付録 B） |
| **FFN（全結合層 2 枚）** | 各トークンが受け取った情報を、単独で加工・記憶する |
| **残差接続（`x + ...`）** | 元の情報を保ったまま変更分だけ足す。層を深く積むための必須テクニック |

順番はこうです。

```
x ──┬──▶ 正規化 ──▶ Attention ──┐
    └───────────────────────────(+)──┬──▶ 正規化 ──▶ FFN ──┐
                                     └──────────────────────(+)──▶ 出力
```

**入力と出力の形が同じ**なので、このブロックは何段でも積み重ねられます。
Llama-3.2-1B はこれを 16 段積んだものです。

In [ ]:
# 実行時間: 数秒

class MultiHeadSelfAttention(nn.Module):
    """因果マスク付き Multi-Head Self-Attention"""

    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads

        # Q・K・V を作るための重み（Part 2 で手で作っていたもの）
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        # 各ヘッドの出力を結合したあと、元の次元に戻す層
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, C = x.shape   # (バッチ, トークン数, 隠れ次元)

        # (B, T, C) → (B, ヘッド数, T, ヘッドの次元) に分割
        def split_heads(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q = split_heads(self.q_proj(x))
        k = split_heads(self.k_proj(x))
        v = split_heads(self.v_proj(x))

        # ここは Part 2 で書いた式とまったく同じ
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)

        # 因果マスク（Part 3）
        mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
        scores = scores.masked_fill(~mask, float('-inf'))

        weights = torch.softmax(scores, dim=-1)
        out = weights @ v                                    # (B, ヘッド数, T, d_head)

        # ヘッドを結合して元の形に戻す
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out), weights


class TransformerBlock(nn.Module):
    """Transformer ブロック 1 段ぶん"""

    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = MultiHeadSelfAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp   = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False),
            nn.SiLU(),                              # 活性化関数（ch1_02 の ReLU の仲間）
            nn.Linear(d_ff, d_model, bias=False),
        )

    def forward(self, x):
        # 残差接続: 元の x に「変更分」を足す
        attn_out, attn_weights = self.attn(self.norm1(x))
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x, attn_weights


print('Transformer ブロックの定義完了（正味 50 行ほど）')

In [ ]:
# 実行時間: 数秒

# 小さめの設定で 1 ブロック作って動かしてみる
demo_block = TransformerBlock(d_model=64, n_heads=4, d_ff=256)

dummy = torch.randn(1, 6, 64)   # バッチ1・6トークン・64次元
out, w = demo_block(dummy)

print(f'入力の形状   : {tuple(dummy.shape)}')
print(f'出力の形状   : {tuple(out.shape)}  ← 入力と同じ！')
print(f'注目度の形状 : {tuple(w.shape)}  (バッチ, ヘッド, 見る側, 見られる側)')
print(f'パラメータ数 : {sum(p.numel() for p in demo_block.parameters()):,}')

# 形が変わらないので、そのまま積み重ねられる
blocks = nn.ModuleList([TransformerBlock(64, 4, 256) for _ in range(4)])

h = dummy
for i, blk in enumerate(blocks):
    h, _ = blk(h)
    print(f'  ブロック {i + 1} 通過後: {tuple(h.shape)}')

print('\n形が変わらない = 何段でも積める。これが「深さ」を稼げる理由。')

## C-1. パラメータはどこに使われているか

本編 Part 4 の対応表で見たとおり、部品の構成は本物とほぼ同じでした。
では、12 億個のパラメータは**どこに**あるのでしょうか。

In [ ]:
# 実行時間: 数秒
# パラメータがどこに使われているかを集計する

buckets = {'埋め込み': 0, 'Attention': 0, 'FFN (MLP)': 0, '正規化': 0, 'その他': 0}

for name, p in model.named_parameters():
    n = p.numel()
    if 'embed' in name or 'lm_head' in name:
        buckets['埋め込み'] += n
    elif 'self_attn' in name:
        buckets['Attention'] += n
    elif 'mlp' in name:
        buckets['FFN (MLP)'] += n
    elif 'norm' in name:
        buckets['正規化'] += n
    else:
        buckets['その他'] += n

total = sum(buckets.values())

fig, ax = plt.subplots(figsize=(8, 3.6))
names = list(buckets.keys())
vals  = [buckets[k] / 1e6 for k in names]
bars = ax.barh(names, vals, color=[GREEN, '#1A6B52', AMBER, '#C5BFB2', '#DDD8CE'])
for bar, k in zip(bars, names):
    ax.text(bar.get_width() + total / 1e6 * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{buckets[k] / 1e6:.0f}M ({buckets[k] / total * 100:.0f}%)',
            va='center', fontsize=9)
ax.set_xlabel('パラメータ数（百万）')
ax.set_title(f'Llama-3.2-1B のパラメータ内訳（合計 {total / 1e9:.2f}B）', fontsize=11)
ax.set_xlim(0, max(vals) * 1.25)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('注目点: Attention より FFN のほうがパラメータを食っている。')
print('「知識を溜め込んでいるのは主に FFN」と言われる根拠のひとつ。')

---

# 付録 D: Attention をもっと覗く

## D-1. 層によって役割が違う

16 層すべてが同じことをしているわけではありません。層ごとの傾向を見てみます。

In [ ]:
# 実行時間: 数秒

fig, axes = plt.subplots(1, 4, figsize=(17, 4.4))
for ax, layer in zip(axes, [0, 3, 8, 15]):
    # その層のヘッド平均を表示
    plot_attention(A[layer].mean(dim=0), tok_labels, tok_labels,
                   f'第{layer}層（全ヘッド平均）', ax=ax, show_values=False)
    ax.set_xlabel('')
    ax.set_ylabel('')
plt.tight_layout()
plt.show()

print('浅い層 : 直前のトークンや文頭を見る、局所的なパターンが多い')
print('中間層 : 単語同士の関係（係り受け・共参照）を捉えるヘッドが増える')
print('深い層 : 次のトークンを出力するための情報統合に向かう')

> **コラム：Attention Sink（注目の吸い込み口）**
>
> 上の図で、**一番左の列（`<|begin_of_text|>` = 文頭トークン）が異様に明るい**ことに
> 気づいたでしょうか。これは実装のバグではなく、よく知られた現象です。
>
> softmax は必ず合計を 1 にするため、「特に見るべき相手がいない」ときでもどこかに注目を
> 割り振らざるを得ません。その捨て場所として、モデルは文頭トークンを使うことを学習します。
> これを **Attention Sink** と呼びます。
>
> 実際にどれくらい吸い込まれているか測ってみましょう。

In [ ]:
# 実行時間: 数秒

# 最後のトークンから見た、文頭トークンへの注目度を全ヘッドで集計
sink = A[:, :, -1, 0]                    # (層数, ヘッド数)

print(f'最終トークンから文頭トークンへの注目度')
print(f'  全 {sink.numel()} ヘッドの平均 : {float(sink.mean()):.3f}')
print(f'  注目度が 0.5 を超えるヘッド  : {int((sink > 0.5).sum())} 個 '
      f'({float((sink > 0.5).float().mean()) * 100:.0f}%)')

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(sink.flatten().numpy(), bins=30, color=GREEN, edgecolor='white')
ax.set_xlabel('文頭トークンへの注目度')
ax.set_ylabel('ヘッド数')
ax.set_title('Attention Sink：多くのヘッドが文頭に注目を捨てている', fontsize=11)
plt.tight_layout()
plt.show()

print('→ Attention マップを読むときは、文頭への注目は「無視」と解釈してよいことが多い。')

---

# 付録 E: 生成の内側

## E-1. 実は全位置ぶんの予測を同時に出している

`logits` の形は `(1, トークン数, 語彙数)` でした。
つまりモデルは最後の位置だけでなく、**入力の各位置について「その次は何か」を同時に予測**しています。

学習時はこれを利用して、1 回の順伝播で全位置ぶんの予測誤差をまとめて計算します。
これが事前学習の効率を支えています。

In [ ]:
# 実行時間: 数秒

text_b = 'The cat sat on the'
enc_b = tokenizer(text_b, return_tensors='pt').to(model.device)
with torch.no_grad():
    logits_b = model(**enc_b).logits[0].float().cpu()

toks_b = [tokenizer.decode([i]) for i in enc_b['input_ids'][0]]

print(pad('位置', 6) + pad('ここまでの入力', 30) + pad('次の予測 1位', 18, right=True)
      + pad('確率', 10, right=True))
print('-' * 64)
for pos in range(1, len(toks_b)):          # 位置 0 は文頭トークンなので飛ばす
    p = torch.softmax(logits_b[pos], dim=-1)
    best = int(p.argmax())
    context = ''.join(toks_b[1:pos + 1])
    print(pad(pos, 6) + pad(context, 30) + pad(repr(tokenizer.decode([best])), 18, right=True)
          + pad(f'{float(p[best]) * 100:.1f}%', 10, right=True))

たった 10 行ほどのループで文章生成ができました。
**`model.generate()` が内部でやっていることも、本質的にはこれと同じです。**

## E-2. KV キャッシュ：同じ計算を繰り返さない

上のループには無駄があります。1 トークン増えるたびに、
**すでに計算済みの過去のトークンの K・V を毎回作り直している**のです。

そこで、過去の K・V を保存しておいて使い回します。これが **KV キャッシュ** です。
LLM 推論のメモリ使用量の大部分は、実はこのキャッシュが占めています。

効果を測ってみましょう。

In [ ]:
# 実行時間: GPU で約10秒

long_prompt = 'AIの歴史について説明します。' * 60
enc_long = tokenizer(long_prompt, return_tensors='pt').to(model.device)
print(f'プロンプト長: {enc_long["input_ids"].shape[1]} トークン\n')

def bench(use_cache: bool, n: int = 60) -> float:
    with torch.no_grad():   # 1回ウォームアップしてから計測
        model.generate(**enc_long, max_new_tokens=5, do_sample=False,
                       use_cache=use_cache, pad_token_id=tokenizer.eos_token_id)
    if device == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        model.generate(**enc_long, max_new_tokens=n, do_sample=False,
                       use_cache=use_cache, pad_token_id=tokenizer.eos_token_id)
    if device == 'cuda':
        torch.cuda.synchronize()
    return time.time() - t0

t_on  = bench(True)
t_off = bench(False)

print(f'KV キャッシュ あり : {t_on:.2f} 秒')
print(f'KV キャッシュ なし : {t_off:.2f} 秒')
print(f'→ {t_off / t_on:.1f} 倍の高速化')
print('\nプロンプトが長いほど差は大きくなる。')
print('長文入力を扱う実運用では、KV キャッシュが速度とメモリの両方を左右する。')

## E-3. サンプリングを自分で実装する

Part 6 の貪欲法ループに、temperature と top_p を組み込みます。
`model.generate()` がやっていることを、そのまま手で書き下したものです。

In [ ]:
# 実行時間: GPU で約15秒

def sample_next_token(logits, temperature: float = 1.0, top_p: float = 1.0) -> int:
    """logits から次のトークンを1つサンプリングする"""
    logits = logits.float()

    # ① temperature で分布の尖りを調整
    probs = torch.softmax(logits / max(temperature, 1e-6), dim=-1)

    # ② top_p で候補を絞る
    if top_p < 1.0:
        sorted_probs, sorted_idx = probs.sort(descending=True)
        cumsum = sorted_probs.cumsum(dim=-1)
        # 累積確率が top_p に達した「次」以降を捨てる
        remove = (cumsum - sorted_probs) >= top_p
        sorted_probs[remove] = 0.0
        probs = torch.zeros_like(probs).scatter_(0, sorted_idx, sorted_probs)
        probs = probs / probs.sum()

    # ③ 確率に従ってサイコロを振る
    return int(torch.multinomial(probs, num_samples=1))


def sampling_generate(text: str, max_new_tokens: int = 30,
                      temperature: float = 1.0, top_p: float = 1.0) -> str:
    """自作サンプリングによる生成（model.generate() は使わない）"""
    ids = tokenizer(text, return_tensors='pt')['input_ids'].to(model.device)
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(ids).logits[0, -1]
        next_id = sample_next_token(logits, temperature, top_p)
        if next_id == tokenizer.eos_token_id:
            break
        ids = torch.cat([ids, torch.tensor([[next_id]], device=ids.device)], dim=1)
    return tokenizer.decode(ids[0], skip_special_tokens=True)


start = 'The weather today is'
print('■ 貪欲法（毎回同じ）')
for i in range(2):
    print(f'  {i + 1}回目: {greedy_generate(start, max_new_tokens=15, verbose=False)}')

print('\n■ 自作サンプリング temperature=1.0, top_p=0.9（毎回違う）')
for i in range(3):
    torch.manual_seed(i)
    print(f'  {i + 1}回目: {sampling_generate(start, max_new_tokens=15, temperature=1.0, top_p=0.9)}')

---

# 付録 F: 推論パラメータを定量的に評価する

本編では temperature を 2 値で比べて「出力が変わる」ことを体感しました。
ここでは **感覚ではなく数値で** 効果を測ります。

## F-1. Temperature の実験

同一プロンプトに対して 3 種類の temperature で生成し、出力の傾向を比較します。

| temperature | 期待される傾向 |
|---|---|
| `0.1` | 確実・再現性高・同じ表現を繰り返しやすい |
| `0.7` | バランス型（多くの場面でのデフォルト） |
| `1.3` | 多様・創造的・ときに意外な表現が出る |

In [ ]:
# 実行時間: GPU で約15秒、CPU で約5分

prompt_temp = build_prompt('AIの未来について、自由に語ってください。')
temperatures = [0.1, 0.7, 1.3]
colors = [GREEN, AMBER, RED]

temp_results = {}

for t in temperatures:
    torch.manual_seed(42)  # 比較のためシード固定
    response = generate_text(prompt_temp, max_new_tokens=120, temperature=t, top_p=0.9)
    temp_results[t] = response
    print(f'\n{"=" * 60}')
    print(f'temperature = {t}')
    print('=' * 60)
    print(response)

### Temperature によるトークン多様性の定量比較

「多様に見える」を感覚で終わらせず、数値にします。
各出力に含まれるユニークトークンの比率（type-token ratio）を使います。

$$\text{TTR} = \frac{\text{ユニークトークン数}}{\text{総トークン数}}$$

In [ ]:
# 実行時間: 数秒

def type_token_ratio(text: str) -> float:
    """テキストのトークン多様性（TTR）を計算する"""
    token_ids = tokenizer.encode(text)
    return len(set(token_ids)) / len(token_ids) if token_ids else 0.0


ttrs = {t: type_token_ratio(resp) for t, resp in temp_results.items()}

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar([str(t) for t in temperatures], list(ttrs.values()), color=colors, width=0.5)
for bar, val in zip(bars, ttrs.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)

ax.set_xlabel('temperature')
ax.set_ylabel('Type-Token Ratio（高いほど多様）')
ax.set_title('Temperature とトークン多様性の関係')
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.show()

for t, ttr in ttrs.items():
    print(f'temperature={t:.1f} | TTR={ttr:.3f}')

## F-2. Top-p の実験

Temperature を固定（`0.9`）し、`top_p` だけを変えて出力の変化を確認します。
Top-p が小さいほど候補が絞られ、確実性が上がります。

In [ ]:
# 実行時間: GPU で約15秒、CPU で約5分

prompt_topp = build_prompt('料理のレシピを1つ考えてください。')
top_ps = [0.5, 0.9, 1.0]

topp_results = {}

for p in top_ps:
    torch.manual_seed(42)
    response = generate_text(prompt_topp, max_new_tokens=120, temperature=0.9, top_p=p)
    topp_results[p] = response
    print(f'\n{"=" * 60}')
    print(f'top_p = {p}')
    print('=' * 60)
    print(response)

## F-3. 出力のばらつきを Jaccard 類似度で測る

同じプロンプト・同じパラメータで 5 回生成し、出力の「ばらつき」をトークン共通率で可視化します。

$$\text{Jaccard}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

- **低 temperature**: どの回も似た出力になる → 共通率が高い
- **高 temperature**: 毎回異なる出力になる → 共通率が低い

In [ ]:
# 実行時間: GPU で約40秒（3条件 × 5回生成）、CPU で約15分

def token_overlap(text_a: str, text_b: str) -> float:
    """2つのテキスト間のトークン集合の Jaccard 類似度を計算する"""
    set_a = set(tokenizer.encode(text_a))
    set_b = set(tokenizer.encode(text_b))
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


N_TRIALS = 5
prompt_var = build_prompt('今日の気分を一文で表現してください。')
test_temps = [0.1, 0.7, 1.3]

all_outputs = {}
for t in test_temps:
    outputs = []
    for i in range(N_TRIALS):
        torch.manual_seed(i)   # 試行ごとにシードを変える
        outputs.append(generate_text(prompt_var, max_new_tokens=60, temperature=t, top_p=0.9))
    all_outputs[t] = outputs
    print(f'temperature={t}: {N_TRIALS}回生成完了')

# 全ペアの Jaccard 類似度の平均
avg_overlaps = {}
for t, outputs in all_outputs.items():
    pairs = list(combinations(outputs, 2))
    avg_overlaps[t] = sum(token_overlap(a, b) for a, b in pairs) / len(pairs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bars = axes[0].bar([str(t) for t in test_temps], list(avg_overlaps.values()),
                   color=colors, width=0.5)
for bar, val in zip(bars, avg_overlaps.values()):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=10)
axes[0].set_xlabel('temperature')
axes[0].set_ylabel('平均 Jaccard 類似度')
axes[0].set_title('出力どうしの似かより（高い＝ばらつきが小さい）', fontsize=11)
axes[0].set_ylim(0, 1.0)

axes[1].bar([str(t) for t in test_temps],
            [1 - v for v in avg_overlaps.values()], color=colors, width=0.5)
axes[1].set_xlabel('temperature')
axes[1].set_ylabel('1 − 平均 Jaccard 類似度')
axes[1].set_title('出力のばらつき（高い＝多様）', fontsize=11)
axes[1].set_ylim(0, 1.0)

plt.tight_layout()
plt.show()

print('\n--- 実際の出力例（各 temperature の1回目）---')
for t in test_temps:
    print(f'\n[temperature={t}] {all_outputs[t][0][:120]}')

## F-4. repetition_penalty の効果

`repetition_penalty` は、**すでに出たトークンの logits を割り引く**パラメータです。
長い出力で「同じ表現の無限ループ」に陥るのを防ぎます。

$$\text{2-gram重複率} = 1 - \frac{\text{ユニーク2-gram数}}{\text{総2-gram数}}$$

In [ ]:
# 実行時間: GPU で約15秒、CPU で約5分

prompt_rep = build_prompt('春について、詩のように語ってください。')
rep_penalties = [1.0, 1.2, 1.5]

for r in rep_penalties:
    torch.manual_seed(42)
    response = generate_text(prompt_rep, max_new_tokens=150,
                             temperature=0.9, top_p=0.92, repetition_penalty=r)

    token_ids = tokenizer.encode(response)
    bigrams   = list(zip(token_ids, token_ids[1:]))
    dup_ratio = 1 - len(set(bigrams)) / max(len(bigrams), 1)

    print(f'\n{"=" * 60}')
    print(f'repetition_penalty = {r}  |  2-gram 重複率: {dup_ratio:.3f}')
    print('=' * 60)
    print(response)

---

# 付録のまとめ

| 付録 | 分かったこと |
|---|---|
| **A** | $\sqrt{d_k}$ は次元数によらずスコアの散らばりを一定に保つ。学習前のランダムな重みでは注目先に意味がない |
| **B** | ヘッドごとに違う関係を見る。実装は $d_{model}$ を分割して並列に走らせる |
| **C** | Transformer ブロックは正味 50 行。入出力の形が変わらないから何段でも積める。**パラメータの大半は Attention ではなく FFN** |
| **D** | 浅い層は局所的、中間層は単語同士の関係を見る。多くのヘッドは文頭トークンに注目を捨てている（Attention Sink） |
| **E** | モデルは全位置ぶんの予測を同時に出している。KV キャッシュは長いプロンプトほど効く |
| **F** | temperature が上がると TTR（多様性）は上がり、出力どうしの Jaccard 類似度は下がる |

本編に戻って `exercises/ex_03_llm_inference.ipynb` に進むか、
第2章の **ファインチューニング（SFT / LoRA）** と **RAG** へ進んでください。